# Data Preparation

Tahap ini menjelaskan proses pengumpulan dan penyiapan data berita yang akan digunakan. Data dikumpulkan dari situs Detik.com dengan menggunakan crawling untuk memperoleh 200 artikel berita.
Crawling adalah proses mengumpulkan data dari halaman web secara otomatis menggunakan program atau script.

1. Mengumpulkan URL artikel secara otomatis dari halaman indeks kategori
2. Mengekstrak isi utama setiap artikel menggunakan Trafilatura
3. Memberi label sesuai kategori
4. Menggabungkan menjadi satu dataset dengan skema `id`, `isi_berita`, `label`, `url`

In [1]:
import sys
!{sys.executable} -m pip install -U lxml_html_clean lxml trafilatura


[notice] A new release of pip available: 22.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import requests
import trafilatura
import pandas as pd
import re
import time

## 1. Fungsi Ekstraksi Isi Artikel

`trafilatura.fetch_url()` terkadang mengembalikan HTML yang tidak sesuai dari server Detik. Solusi yang stabil: mengambil HTML sendiri menggunakan `requests`, baru diteruskan ke `trafilatura.extract()`.

In [3]:
def ambil_isi_berita(url):
    """Mengambil isi utama sebuah artikel berita dari URL menggunakan requests + trafilatura."""
    try:
        resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
        if resp.status_code == 200:
            teks = trafilatura.extract(resp.text, url=url)
            return teks
        else:
            return None
    except Exception as e:
        print(f"Error: {e} - {url}")
        return None

### Demonstrasi: HTML Mentah vs Hasil Ekstraksi

Berikut perbandingan antara HTML mentah sebuah halaman artikel dengan hasil ekstraksi menggunakan Trafilatura, untuk menunjukkan bahwa proses ekstraksi benar-benar menyaring konten utama dari elemen-elemen lain seperti menu, iklan, dan navigasi.

In [4]:
url_demo = "https://sport.detik.com/moto-gp/d-8652587/motogp-san-marino-antusiasme-bezzecchi-balapan-di-rumah-sendiri"

resp_demo = requests.get(url_demo, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
html_mentah = resp_demo.text
isi_bersih = trafilatura.extract(html_mentah, url=url_demo)

print("--- HTML MENTAH (potongan 500 karakter pertama) ---")
print(html_mentah[:500])
print(f"\nTotal panjang HTML mentah: {len(html_mentah):,} karakter\n")

print("--- HASIL EKSTRAKSI TRAFILATURA (potongan 500 karakter pertama) ---")
print(isi_bersih[:500])
print(f"\nTotal panjang hasil ekstraksi: {len(isi_bersih):,} karakter")

--- HTML MENTAH (potongan 500 karakter pertama) ---
<!DOCTYPE html>
<html lang="id-ID">
    <head>
        <script type="text/javascript" src="https://iat.detiknetwork.com/ip-information.js"></script>
        <link rel="preconnect" href="https://awscdn.detik.net.id"><link rel="preconnect" href="https://awscdn.detik.net.id" crossorigin><link rel="preconnect" href="https://cdn.detik.net.id"><link rel="preconnect" href="https://cdn.detik.net.id" crossorigin>        <link rel="dns-prefetch" href="https://cdn.detik.net.id"/><link rel="dns-prefetch" h

Total panjang HTML mentah: 251,538 karakter

--- HASIL EKSTRAKSI TRAFILATURA (potongan 500 karakter pertama) ---
Marco Bezzecchi tak sabar menghadapi MotoGP San Marino 2026. Bagi pebalap Aprilia itu, balapan di Sirkuit Misano punya arti khusus karena lokasinya sangat dekat dengan kampung halamannya, Rimini.
MotoGP San Marino akan berlangsung di Misano pada 11-13 September. Bezzecchi menyebut seri ini selalu terasa berbeda dibandingkan balapan 

Terlihat jelas bahwa HTML mentah (~250.000+ karakter) mengandung banyak kode markup, skrip, dan elemen navigasi yang tidak relevan, sedangkan hasil ekstraksi Trafilatura (~1.800 karakter) hanya berisi teks inti artikel yang siap digunakan untuk analisis.

## 2. Fungsi Pengumpulan URL Artikel

Detik.com menyediakan halaman indeks kategori (contoh: `sport.detik.com/indeks`) yang dapat dipaginasi dengan parameter `?page=N`. Semua tautan artikel mengikuti pola `https://<subdomain>.detik.com/<subkanal>/d-<nomor>/<judul-slug>`, sehingga dapat diambil otomatis menggunakan regex.

In [5]:
def ambil_url_dari_halaman_index(halaman_url):
    """Mengambil semua URL artikel dari 1 halaman index Detik."""
    resp = requests.get(halaman_url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
    html = resp.text
    pola = r"https://[a-z]+\.detik\.com/[a-z0-9-]+/d-\d+/[a-z0-9-]+"
    urls = re.findall(pola, html)
    return list(dict.fromkeys(urls))


def kumpulkan_url_kategori(base_index_url, target_jumlah=100, max_halaman=15):
    """Mengumpulkan URL artikel dari banyak halaman index sampai mencapai target_jumlah."""
    semua_url = []
    halaman = 1
    while len(semua_url) < target_jumlah and halaman <= max_halaman:
        url_halaman = f"{base_index_url}?page={halaman}"
        url_baru = ambil_url_dari_halaman_index(url_halaman)
        for u in url_baru:
            if u not in semua_url:
                semua_url.append(u)
        halaman += 1
        time.sleep(1)
    return semua_url[:target_jumlah]

Kode tersebut merupakan bagian dari proses web crawling, khususnya tahap pengumpulan URL artikel dari halaman indeks Detik.com. Fungsi ambil_url_dari_halaman_index() digunakan untuk mengambil URL artikel dari satu halaman indeks, sedangkan kumpulkan_url_kategori() mengumpulkan URL dari beberapa halaman indeks hingga mencapai jumlah yang ditentukan.

Dengan proses tersebut, URL artikel dapat dikumpulkan secara otomatis, menghindari duplikasi, dan digunakan sebagai sumber untuk tahap berikutnya, yaitu mengambil isi artikel dan membentuk dataset berita

## 3. Proses Crawling & Penyusunan Dataset

Untuk menghindari crawling berulang yang membebani server Detik.com, proses ini menggunakan flag `_re_crawl`. Jika `False` (default), notebook akan memuat dataset yang sudah tersimpan di `data/dataset_berita_200.xlsx`. Jika ingin menjalankan ulang crawling dari awal, ubah menjadi `True`.

In [6]:
_re_crawl = False  # ubah ke True jika ingin crawling ulang dari awal

def crawl_kategori(base_index_url, label, target_jumlah=100):
    """Crawling penuh 1 kategori: kumpulkan URL, ekstrak isi, beri label."""
    urls = kumpulkan_url_kategori(base_index_url, target_jumlah)
    hasil = []
    gagal = []
    for u in urls:
        isi = ambil_isi_berita(u)
        if isi and len(isi) > 100:
            hasil.append({"isi_berita": isi, "label": label, "url": u})
        else:
            gagal.append(u)
        time.sleep(1)

    # retry sekali untuk yang gagal
    for u in gagal[:]:
        isi = ambil_isi_berita(u)
        if isi and len(isi) > 100:
            hasil.append({"isi_berita": isi, "label": label, "url": u})
            gagal.remove(u)

    return hasil, gagal

In [7]:
import sys

!{sys.executable} -m pip install openpyxl


[notice] A new release of pip available: 22.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
if _re_crawl:
    data_sport, gagal_sport = crawl_kategori("https://sport.detik.com/indeks", "sport", 100)
    data_finance, gagal_finance = crawl_kategori("https://finance.detik.com/indeks", "finance", 100)

    rows = []
    for i, item in enumerate(data_sport, start=1):
        rows.append({"id": i, "isi_berita": item["isi_berita"], "label": item["label"], "url": item["url"]})
    for i, item in enumerate(data_finance, start=101):
        rows.append({"id": i, "isi_berita": item["isi_berita"], "label": item["label"], "url": item["url"]})

    df = pd.DataFrame(rows)
    df.to_excel("../data/dataset_berita_200.xlsx", index=False)
    print("Crawling selesai, dataset baru disimpan.")
else:
    df = pd.read_excel("../data/dataset_berita_200.xlsx")
    print("Dataset dimuat dari file tersimpan (tidak melakukan crawling ulang).")

print("Total baris:", len(df))
df.head(3)

Dataset dimuat dari file tersimpan (tidak melakukan crawling ulang).
Total baris: 200


,id,isi_berita,label,url
0,1,Chef de Mission (CdM) Indonesia Todotua Pasari...,sport,https://sport.detik.com/sport-lain/d-8655308/c...
1,2,"Banjir besar melanda Nagoya, Jepang, menjelang...",sport,https://sport.detik.com/sport-lain/d-8655303/a...
2,3,Ketua Umum Komite Olimpiade Indonesia (KOI) Ra...,sport,https://sport.detik.com/sport-lain/d-8655067/k...


Kode ini digunakan untuk menghasilkan dataset berita sebanyak 200 artikel dari hasil crawling Detik.com, terdiri dari 100 artikel Sport dan 100 artikel Finance, kemudian menyimpannya ke Excel. Jika dataset sudah tersedia dan tidak ingin crawling ulang, program akan langsung membaca file Excel tersebut.

## 4. Validasi Dataset Akhir

In [ ]:
assert len(df) == 200, "Total baris harus 200"
assert (df["label"] == "sport").sum() == 100, "Jumlah sport harus 100"
assert (df["label"] == "finance").sum() == 100, "Jumlah finance harus 100"
assert df["isi_berita"].isna().sum() == 0, "Tidak boleh ada isi_berita kosong"
assert list(df.columns) == ["id", "isi_berita", "label", "url"], "Skema kolom harus id, isi_berita, label, url"

print("Semua validasi berhasil. Dataset siap digunakan untuk tahap Modeling.")

Semua validasi berhasil. Dataset siap digunakan untuk tahap Modeling.
